# 200. Number of Islands
**Difficulty:** 🟡 Medium · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/number-of-islands/

## 💡 Concepts

**Core concept(s):** **Flood fill** (DFS or BFS) on a grid.

**Why it applies here:** Each island is a connected blob of land. Scan the grid; when you hit unvisited land, flood-fill the whole blob (sinking it so you don't recount it) and add one to the island count.

**Key intuition:** Find a piece of land, drown the entire connected island, and tally one — repeat.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is Flood Fill?
**Flood fill** is DFS/BFS on a grid: from a starting cell, spread to matching neighbors (up/down/left/right), marking them visited — like the paint-bucket tool.
- **Complexity:** **O(rows × cols)** — each cell handled once.

### 📚 What is DFS (Depth-First Search)?
**DFS** follows one path as deep as it goes, then backtracks. On graphs you must remember **visited** nodes so you don't loop forever.
- **Complexity:** **O(V + E)** — each node and edge once.
- **In Python:** recursion or an explicit stack, plus a `visited` set.

### 📚 What is BFS (Breadth-First Search)?
**BFS** explores in rings outward from the start using a **queue**, visiting nearer nodes first.
- **Complexity:** **O(V + E)**; great for shortest number of steps.
- **In Python:** `collections.deque` plus a `visited` set.

---

**Prerequisite knowledge:**
- Grid traversal.
- Marking visited cells.

## 📝 Problem

Count the islands in a grid of `'1'` (land) and `'0'` (water). Land connects horizontally/vertically.

**Example**
```
11110
11010
11000
00000   -> 1 island
```

> Two approaches, both `O(rows×cols)`: DFS flood fill and BFS flood fill. (Union-Find also works.)

### Approach 1 — DFS Flood Fill

**Idea:** For each land cell not yet sunk, DFS across its whole island, sinking every cell; count one island.

**Time:** `O(m·n)`. **Space:** `O(m·n)` worst-case recursion.

In [ ]:
def num_islands_dfs(grid):
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    g = [row[:] for row in grid]           # work on a copy so we don't alter the caller's grid
    def dfs(r, c):                         # sink this whole connected island
        if r < 0 or c < 0 or r >= rows or c >= cols or g[r][c] != "1":
            return                         # off the grid or water -> stop
        g[r][c] = "0"                      # mark this land cell visited (sink it)
        dfs(r+1,c); dfs(r-1,c); dfs(r,c+1); dfs(r,c-1)   # spread to 4 neighbors
    count = 0
    for r in range(rows):
        for c in range(cols):
            if g[r][c] == "1":             # found a new, unvisited island
                count += 1; dfs(r, c)      # count it and sink the whole thing
    return count

### Approach 2 — BFS Flood Fill

**Idea:** Same, but sink each island with a queue instead of recursion (safer for huge islands).

**Time:** `O(m·n)`. **Space:** `O(min(m,n))` for the queue.

In [ ]:
from collections import deque

def num_islands_bfs(grid):
    if not grid or not grid[0]:
        return 0
    rows, cols = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    count = 0
    for r in range(rows):
        for c in range(cols):
            if g[r][c] == "1":             # start of a new island
                count += 1
                g[r][c] = "0"; q = deque([(r, c)])   # sink it using a queue (BFS)
                while q:
                    x, y = q.popleft()
                    for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
                        nx, ny = x+dx, y+dy
                        if 0 <= nx < rows and 0 <= ny < cols and g[nx][ny] == "1":
                            g[nx][ny] = "0"; q.append((nx, ny))   # sink and queue neighbors
    return count

In [ ]:
# Correctness check
g1 = [list("11110"),list("11010"),list("11000"),list("00000")]
g2 = [list("11000"),list("11000"),list("00100"),list("00011")]
tests = [(g1,1), (g2,3), ([list("000")],0)]
for grid, exp in tests:
    a, b = num_islands_dfs(grid), num_islands_bfs(grid)
    print(f"-> dfs={a}, bfs={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # checkerboard: many 1-cell islands -> shallow recursion, full O(mn) work
    grid = [["1" if (r + c) % 2 == 0 else "0" for c in range(n)] for r in range(n)]
    return (grid,)
solutions = {
    "dfs O(mn)": num_islands_dfs,
    "bfs O(mn)": num_islands_bfs,
}
sizes = [50, 100, 200, 300]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Flood fill counts blobs:** scan, and each time you meet unvisited land, sink the whole connected region and count one.
- **Sink as you go:** overwriting visited land avoids a separate visited set.
- **Signal:** "connected regions / islands / groups on a grid".
- **Related problems:** Max Area of Island, Surrounded Regions, Pacific Atlantic, Number of Connected Components.
- **Common pitfalls:** (1) recounting a region (mark visited immediately); (2) very large single islands overflowing DFS recursion (use BFS).